# AeroFleet — 1,000-case mass forensics run on Kaggle (free GPU)

Runs the real (non-mock) LLM evaluation harness against a free Kaggle GPU, chunked across
sessions using the harness's own per-case checkpoint (`results.jsonl`). Safe to stop and resume —
each session picks up exactly where the last one left off.

This version runs the harness as a **background process** instead of a blocking cell, so you can
check progress / `ollama ps` / logs at any time without waiting, and stop it cleanly without risk
of also killing the Ollama server alongside it.

## Required one-time manual setup (Kaggle UI, not this notebook)

1. **Settings → Accelerator → GPU T4 x2** (or whatever GPU option Kaggle currently offers —
   the exact name has changed before). Settings → Internet → **On**. If Accelerator options are
   missing/greyed out, verify your phone number first: profile icon → Settings → Phone Verification.
2. **Add-ons → Secrets → add a secret named `GH_PAT`**: a GitHub fine-grained Personal Access
   Token, read-only, scoped to just this repo (`AdityaPathare46/aerofleet`, private). Toggle it
   **on** for this notebook specifically — creating the secret alone isn't enough.
3. Session cap is several hours and Kaggle gives a weekly GPU-hour quota. When a session ends,
   **Save Version** (use the option that saves your *current* session state, not the one that
   re-runs everything from scratch) before closing — that persists `/kaggle/working` (including
   `results.jsonl`) as this notebook's Output for next session to resume from.

## Splitting the work across machines

`--target 1000 --batch-size 25` gives 40 batches. This notebook is scoped to **batches 1-14**
(`ONLY_BATCHES` in the run cell) so it can run in parallel with
`research/colab_mass_forensics_run.ipynb` (batches 27-40) and a third machine (batches 15-26)
without redoing each other's work:

| Machine | Batches | Cases |
|---|---|---|
| **This Kaggle notebook** | **1-14** | **MFI-00001 – MFI-00350** |
| Colab | 27-40 | MFI-00651 – MFI-01000 |
| Third machine | 15-26 | MFI-00351 – MFI-00650 |

Running solo? Set `ONLY_BATCHES = None` in the run cell instead.

## Known, deliberate deviation from the real (college-PC) run — put this in the paper

`llama4:scout` is 67GB (109B-param MoE) and doesn't fit any free-tier GPU. It powers 4 agents:
DISPATCHER, AIRSPACE_SAFETY, AI_VALIDATOR, CONTINGENCY. For this cloud run only, those 4 agents
use **`mistral-nemo:12b`** instead, via the existing `AGENT_MODEL_<AGENT_ID>` env-var override
(`aerofleet/agents/factory.py`) — no code changes. This notebook writes a `cloud_run_metadata.json`
recording the substitution so it's traceable. **All machines in the split above must use this
identical substitution** — otherwise different subsets of the 1,000 cases end up evaluated by
different model configurations, which can't be honestly pooled into one "real LLM" result.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 1. Clone the private repo

Uses the `GH_PAT` secret — the token is interpolated into the clone URL and never printed or
stored in this notebook.


In [ ]:
from kaggle_secrets import UserSecretsClient
_token = UserSecretsClient().get_secret("GH_PAT")

REPO_DIR = "/kaggle/working/aerofleet"
import os
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://{_token}@github.com/AdityaPathare46/aerofleet.git {REPO_DIR}
else:
    print("Repo already present — pulling latest instead of a fresh clone.")
    !cd {REPO_DIR} && git pull

del _token  # don't leave it bound in the kernel namespace longer than needed
%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 2. Install Ollama and start the server in this session

`OLLAMA_KEEP_ALIVE=-1` tells Ollama to never voluntarily unload a model due to idle time — one of
two possible causes (the other being genuine VRAM capacity) of a model getting evicted and
reloaded between agent calls, which would show up as a few-minutes-long gap in the logs for no
obvious reason. Costs nothing to set; only helps if idle-eviction was part of the problem.


In [ ]:
# zstd: required by the Ollama installer to extract its archive.
# pciutils (lspci): lets the installer auto-detect the GPU and install the
# matching CUDA runtime — without it, install silently warns and may fall
# back to a CPU-only build, which would make the run far too slow to finish.
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time, requests, os

os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

log = open("/kaggle/working/ollama_serve.log", "a")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=5)
        print("Ollama server is up.")
        break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not come up — check /kaggle/working/ollama_serve.log")

time.sleep(2)
!grep -i -E "gpu|cuda|library=" /kaggle/working/ollama_serve.log | tail -5


## 3. Pull the models

Three real roster models plus the `mistral-nemo:12b` substitute (~41GB total). If a pull fails or
this cell is interrupted, just re-run it — Ollama resumes partial downloads.


In [ ]:
for model in ["gemma4:12b", "phi4-reasoning:plus", "mistral-small3.2", "mistral-nemo:12b"]:
    print(f"--- pulling {model} ---")
    !ollama pull {model}

!ollama list


## 4. Resume from a previous session's checkpoint (if any)

If you saved a prior session's output and re-attached it as an input dataset (Add Data → Your
Work → this notebook's earlier Output), point `PREVIOUS_RESULTS_DIR` at the mounted path Kaggle
shows in the file browser (something like `/kaggle/input/<notebook-slug>/mass_forensics_kaggle`) —
its `results.jsonl` gets copied into this session's study dir so the harness resumes instead of
starting over. Leave as `None` for the very first session.


In [ ]:
import shutil
from pathlib import Path

STUDY_DIR = Path("/kaggle/working/mass_forensics_kaggle")
STUDY_DIR.mkdir(parents=True, exist_ok=True)

PREVIOUS_RESULTS_DIR = None  # e.g. "/kaggle/input/<notebook-slug>/mass_forensics_kaggle"

if PREVIOUS_RESULTS_DIR is not None:
    prev = Path(PREVIOUS_RESULTS_DIR)
    for fname in ["results.jsonl", "run_config.json", "dataset_manifest.json"]:
        src = prev / fname
        if src.exists():
            shutil.copy(src, STUDY_DIR / fname)
            print(f"Restored {fname} from previous session")
        else:
            print(f"{fname} not found at {src} — check PREVIOUS_RESULTS_DIR is correct")
else:
    print("No previous session pointed to — starting a fresh study.")


## 5. Configure the model substitution and record it

`AGENT_MODEL_<AGENT_ID>` overrides `AgentFactory.DEFAULT_MODEL_MAP` per-agent, no code edits
needed (`aerofleet/agents/factory.py`).


In [ ]:
import os, json, subprocess, datetime

os.environ["OLLAMA_HOST"] = "http://localhost:11434"
os.environ.pop("USE_MOCK_AGENTS", None)  # make sure we are NOT in mock mode

for agent_id in ["DISPATCHER", "AIRSPACE_SAFETY", "AI_VALIDATOR", "CONTINGENCY"]:
    os.environ[f"AGENT_MODEL_{agent_id}"] = "mistral-nemo:12b"

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()

metadata = {
    "platform": "kaggle",
    "gpu": gpu_name,
    "only_batches": "1-14",
    "run_started_utc": datetime.datetime.utcnow().isoformat(),
    "substitution": {
        "replaced_model": "llama4:scout",
        "substitute_model": "mistral-nemo:12b",
        "reason": "llama4:scout is 67GB (109B-param MoE); does not fit a free-tier GPU",
        "affected_agents": ["DISPATCHER", "AIRSPACE_SAFETY", "AI_VALIDATOR", "CONTINGENCY"],
    },
    "unaffected_agents_real_roster": {
        "ROUTE": "mistral-small3.2", "COMMS": "mistral-small3.2", "COMPLIANCE": "mistral-small3.2",
        "BATTERY": "phi4-reasoning:plus", "COST": "phi4-reasoning:plus", "PAYLOAD": "phi4-reasoning:plus",
        "WEATHER": "gemma4:12b", "OPS": "gemma4:12b",
    },
}
with open(STUDY_DIR / "cloud_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


## 6. Start the harness — runs in the background

This does NOT block the notebook — it starts the run and returns immediately. Use the cells below
to check on it, any time, as often as you like, without needing to stop it first.


In [ ]:
import subprocess, os

ONLY_BATCHES = "1-14"  # this machine's assigned, non-overlapping slice — see the table above.
# Running solo? Set ONLY_BATCHES = None instead.

harness_log_path = STUDY_DIR / "harness_run.log"
harness_log = open(harness_log_path, "a")

cmd = [
    "python", "-m", "scenario_engine.mass_forensics_evaluation",
    "--study-dir", str(STUDY_DIR), "--target", "1000", "--batch-size", "25",
]
if ONLY_BATCHES:
    cmd += ["--only-batches", ONLY_BATCHES]

harness_proc = subprocess.Popen(cmd, stdout=harness_log, stderr=subprocess.STDOUT, env=os.environ.copy())
print(f"Harness started in the background, PID {harness_proc.pid}.")
print(f"Logging to {harness_log_path}")
print("Run the cells below any time to check progress — no need to wait or stop this first.")


## 7. Monitoring — run any of these any time, as often as you like


In [ ]:
# Tail the live log
!tail -n 40 {harness_log_path}


In [ ]:
# Is it still running? (None = still running; a number = exit code, it finished/stopped)
print("exit code (None = still running):", harness_proc.poll())


In [ ]:
# How many case-attempts recorded so far
results_file = STUDY_DIR / "results.jsonl"
if results_file.exists():
    with open(results_file) as f:
        n = sum(1 for _ in f)
    print(f"{n} case-attempts recorded so far (target: 1000 cases; retries count as separate attempts).")
else:
    print("No results yet.")


In [ ]:
# Thrashing check: if this shows a model, then a few cells later shows a DIFFERENT model
# (or nothing) repeatedly, models are being evicted and reloaded between calls — worth flagging.
!ollama ps


## 8. Stopping cleanly

Use this instead of any Kaggle "stop cell" UI button — it terminates only the harness process,
leaving the Ollama server running undisturbed (a UI-level interrupt may not make that distinction
and can take the server down with it, which happened in an earlier session).


In [ ]:
harness_proc.terminate()
harness_proc.wait(timeout=30)
print("Harness stopped cleanly. results.jsonl has everything completed up to this point.")


## 9. End of session — persist for next time

Click **Save Version** (use the option that saves your *current* session state — not the one that
re-runs every cell from scratch, which would restart the whole setup). That snapshots everything
under `/kaggle/working` (including `results.jsonl` and `cloud_run_metadata.json`) as this
notebook's Output. Next session: **Add Data → Your Work → (this notebook, latest version)**, then
set `PREVIOUS_RESULTS_DIR` in the resume cell to the mounted path Kaggle shows for it, and re-run
from the top. Repeat until this machine's assigned batch range (1-14) is complete.

## 10. Once all machines are done (or you want an interim pooled check)

Pull each machine's `results.jsonl` + `dataset_manifest.json` + `run_config.json` to one place and
run `research/merge_batched_results.py` to pool them, then `--report-only` on the merged study —
see `research/colab_mass_forensics_run.ipynb`'s final cell for the exact commands.
